# Projeto de assistente de pedido do Mc Donalds
Um assistente conversacional em português que ajuda a montar um pedido do McDonald's. Ele entende restrições escritas livremente ("frango com baixo sódio", "eu não como carne de vaca"). Se o usuário não disser nenhuma, o assistente pergunta. Os produtos e valores nutricionais vêm do `menu.csv`, não da memória do modelo.

**Limites dos dados.** O `menu.csv` (carregado da branch `main`) é um conjunto histórico do cardápio dos **EUA**. Ele traz nome, categoria, porção e valores nutricionais, mas **não** traz ingredientes, alergênicos nem informação sobre contaminação cruzada. Por isso:

- restrições como "não como carne bovina" são aplicadas de forma **conservadora**, usando apenas o que o nome e a categoria do produto indicam. Um produto sem indicação no nome **não** é declarado livre do ingrediente; ele aparece como "composição não verificada";
- o assistente **nunca** afirma que um item é seguro para alergias ou intolerâncias.

**Como executar.** Rode as células em ordem (Ambiente de execução → Executar tudo). A conversa **não** começa sozinha: ao final, execute `conversar()` na última célula.

### Instalações

In [ ]:
!pip install -q langchain langchain_openai nltk scikit-learn

### Chave e modelo
A chave é lida, nesta ordem, da variável de ambiente `OPENAI_API_KEY`, dos *Secrets* do Colab (ícone de chave na barra lateral) ou digitada com `getpass`. A chave nunca é impressa.

O modelo é usado **apenas** para interpretar a mensagem. Comandos inequívocos (ver pedido, adicionar/remover por ID, perguntas de nutriente sobre um item) são resolvidos localmente e continuam funcionando se o modelo falhar.

In [ ]:
# @colab-only: pede credencial interativamente; os testes automatizados não executam esta célula.
import os
from getpass import getpass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        OPENAI_API_KEY = None
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass("OPENAI_API_KEY: ")

In [ ]:
# @colab-only: cria o cliente do modelo e faz UMA chamada de teste (consome créditos).
from langchain_openai import ChatOpenAI

modelo = ChatOpenAI(model="gpt-6-luna", api_key=OPENAI_API_KEY, timeout=60, max_retries=1)
try:
    print(modelo.invoke("Responda apenas 'ok' se estiver funcionando.").content)
except Exception as erro:
    print("Aviso: o modelo não respondeu:", type(erro).__name__,
          "- o assistente ainda responde comandos locais, mas a interpretação livre fica limitada.")

### Carregar o corpus
Baixa o `menu.csv` da branch `main` (fonte compartilhada do grupo) e valida se o arquivo não está vazio.

In [ ]:
import requests
import pandas as pd
import numpy as np
from io import StringIO

URL_MENU = "https://raw.githubusercontent.com/BeatrizGiacon/Projeto-Pratico-PLN-/main/menu.csv"

def carregar_menu_bruto(url=URL_MENU, timeout=30):
    """Baixa o CSV e devolve o DataFrame bruto. Falha com mensagem clara se o arquivo estiver vazio."""
    resposta = requests.get(url, timeout=timeout)
    resposta.raise_for_status()
    texto = resposta.content.decode("utf-8-sig")
    if not texto.strip():
        raise ValueError("O menu.csv baixado está vazio.")
    bruto = pd.read_csv(StringIO(texto))
    if bruto.empty:
        raise ValueError("O menu.csv não tem linhas de dados.")
    return bruto

df_bruto = carregar_menu_bruto()
print(df_bruto.shape)
df_bruto.head()

In [ ]:
RENOMEAR = {
    "Category": "categoria", "Item": "item", "Serving Size": "porcao_texto",
    "Calories": "calorias", "Total Fat": "gordura_g", "Saturated Fat": "gordura_sat_g",
    "Cholesterol": "colesterol_mg", "Sodium": "sodio_mg", "Carbohydrates": "carbo_g",
    "Sugars": "acucar_g", "Protein": "proteina_g", "Calcium (% Daily Value)": "calcio_vd_pct",
}
COLUNAS_NUMERICAS = ["calorias", "gordura_g", "gordura_sat_g", "colesterol_mg", "sodio_mg",
                     "carbo_g", "acucar_g", "proteina_g", "calcio_vd_pct"]

def preparar_menu(bruto):
    """Seleciona e renomeia colunas, valida tipos e cria id_menu = posição da linha (0..n-1).

    Valores numéricos ausentes, não numéricos, negativos ou infinitos viram NaN
    ("não informado"); eles nunca são tratados como zero.
    """
    if bruto is None or len(bruto) == 0:
        raise ValueError("menu.csv sem linhas.")
    faltantes = [c for c in RENOMEAR if c not in bruto.columns]
    if faltantes:
        raise ValueError(f"Colunas ausentes no menu.csv: {faltantes}")
    tabela = bruto[list(RENOMEAR)].rename(columns=RENOMEAR).copy()
    tabela = tabela[tabela["item"].notna()].copy()
    tabela["item"] = tabela["item"].astype(str).str.replace(r"[®™]", "", regex=True).str.strip()
    tabela["categoria"] = tabela["categoria"].fillna("Sem categoria").astype(str).str.strip()
    tabela["porcao_texto"] = tabela["porcao_texto"].fillna("").astype(str).str.strip()
    tabela = tabela[tabela["item"] != ""].copy()
    for coluna in COLUNAS_NUMERICAS:
        valores = pd.to_numeric(tabela[coluna], errors="coerce").astype(float)
        tabela[coluna] = valores.where(np.isfinite(valores) & (valores >= 0))
    if tabela.empty:
        raise ValueError("menu.csv não contém itens válidos.")
    tabela = tabela.reset_index(drop=True)
    tabela.insert(0, "id_menu", range(len(tabela)))
    # Peso/volume da porção quando o texto informa; bebidas em "fl oz" ficam sem valor.
    tabela["porcao_g"] = tabela["porcao_texto"].str.extract(r"\((\d+(?:\.\d+)?) g\)", expand=False).astype(float)
    tabela["porcao_ml"] = tabela["porcao_texto"].str.extract(r"\((\d+(?:\.\d+)?) ml\)", expand=False).astype(float)
    return tabela

df = preparar_menu(df_bruto)
print("Itens:", len(df), "| nomes repetidos:", int(df["item"].duplicated().sum()),
      "| valores numéricos não informados:", int(df[COLUNAS_NUMERICAS].isna().sum().sum()))
df.head()

### Preparar o texto do menu

Os nomes dos itens ("Big Mac", "Chicken McNuggets") são o único texto do corpus. Este passo segue o tratamento de:

- Normalização (minúsculas, sem acentos) e tokenização.
- Remover palavras vazias como "the" e "of" (stopwords) e faz lematização.
- Criar etiquetas úteis.

Se os recursos do NLTK não puderem ser baixados, a célula usa uma lista mínima de stopwords e segue sem lematização (os tokens são apenas ilustrativos; a busca usa TF-IDF).

In [ ]:
import re, unicodedata

def normalizar(texto):
    """Minúsculas, sem acentos e com espaços simples. None/NaN viram texto vazio."""
    if texto is None or (isinstance(texto, float) and np.isnan(texto)):
        return ""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"\s+", " ", t).strip()

try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    for recurso in ("stopwords", "wordnet", "omw-1.4"):
        nltk.download(recurso, quiet=True)
    stop_en = set(stopwords.words("english"))
    lematizador = WordNetLemmatizer()
    lematizador.lemmatize("tests")
except Exception as erro:
    print("NLTK indisponível (", type(erro).__name__, "): usando stopwords mínimas e sem lematização.")
    stop_en = {"the", "of", "and", "with", "a", "an", "in", "on", "for", "to"}
    lematizador = None

def preprocessar_nome(texto):
    """Normaliza, tokeniza, remove stopwords e lematiza um nome de item."""
    tokens = [t for t in re.findall(r"[a-z]+", normalizar(texto)) if t not in stop_en and len(t) > 2]
    return [lematizador.lemmatize(t) for t in tokens] if lematizador else tokens

df["tokens"] = df["item"].apply(preprocessar_nome)
df[["item", "tokens"]].head(10)

### Documentação para busca semântica
Transforma cada linha em um Document do LangChain, com um texto descritivo e metadados (incluindo `id_menu`). É a base do RAG.

In [ ]:
from langchain_core.documents import Document

def _valor_txt(x, unidade=""):
    return "não informado" if pd.isna(x) else f"{x:.0f}{(' ' + unidade) if unidade else ''}"

def descrever(linha):
    return (f"{linha.item} (categoria: {linha.categoria}). Calorias: {_valor_txt(linha.calorias)}; "
            f"sódio: {_valor_txt(linha.sodio_mg, 'mg')}; gordura: {_valor_txt(linha.gordura_g, 'g')}; "
            f"açúcar: {_valor_txt(linha.acucar_g, 'g')}; proteína: {_valor_txt(linha.proteina_g, 'g')}.")

def criar_documentos(tabela):
    return [Document(page_content=descrever(l),
                     metadata={"id_menu": int(l.id_menu), "item": l.item, "categoria": l.categoria})
            for l in tabela.itertuples()]

documentos = criar_documentos(df)
print(len(documentos), "documentos. Exemplo:", documentos[0].page_content)

**Busca por similaridade de textos**

Usamos TF-IDF com similaridade de cosseno, sem depender de embeddings externos. A busca usa apenas linhas do CSV e é reproduzível. O ranking por relevância é separado da aplicação de restrições: as restrições removem candidatos **antes** do ranking, e o ranking nunca reintroduz um item removido.

In [ ]:
from dataclasses import dataclass
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

@dataclass
class IndiceBusca:
    vetorizador: TfidfVectorizer
    matriz: object
    ids: list

def construir_indice(tabela, documentos):
    """Cria o índice TF-IDF. Exige que documentos e tabela estejam na mesma ordem de id_menu."""
    if [d.metadata["id_menu"] for d in documentos] != tabela["id_menu"].tolist():
        raise ValueError("Documentos e tabela desalinhados: recrie os documentos a partir da tabela atual.")
    corpus = [f"{d.metadata['item']} {d.metadata['categoria']} {d.page_content}" for d in documentos]
    vetorizador = TfidfVectorizer(strip_accents="unicode", lowercase=True, ngram_range=(1, 2), stop_words="english")
    return IndiceBusca(vetorizador, vetorizador.fit_transform(corpus), tabela["id_menu"].tolist())

def buscar_similares(indice, consulta, ids_permitidos=None, k=8):
    """Retorna [(id_menu, escore)] apenas entre ids_permitidos, ordenado por similaridade cosseno."""
    ids = list(indice.ids) if ids_permitidos is None else [int(i) for i in ids_permitidos]
    if not ids or not str(consulta or "").strip():
        return []
    if min(ids) < 0 or max(ids) >= indice.matriz.shape[0]:
        raise IndexError("ID fora do índice TF-IDF: reconstrua a base.")
    pontuacoes = cosine_similarity(indice.vetorizador.transform([consulta]), indice.matriz[ids]).ravel()
    ordem = np.argsort(-pontuacoes, kind="stable")
    return [(ids[j], float(pontuacoes[j])) for j in ordem[:k]]

indice = construir_indice(df, documentos)
print("Exemplo de busca:", [(df.iloc[i]['item'], round(s, 3)) for i, s in buscar_similares(indice, 'chicken nuggets', k=5)])

2. Classificação de intenção e extração de restrições

O LLM recebe instruções para converter a frase livre em uma estrutura validada pelo Pydantic. A classificação identifica intenções como recomendar, adicionar, remover, visualizar pedido e perguntar.

Para tornar termos como “baixo sódio” operacionais, usamos um ranking relativo (menor teor de sódio entre os candidatos), não uma certificação de produto com baixo sódio. Limites numéricos são filtros rígidos e respeitam o operador: “menos de 500” exclui 500; “até 500” inclui 500.

Campos de preferência usam três estados: `None` (não mencionado), `True` (pedido) e `False` (revogado). Valores numéricos precisam ser finitos e não negativos.

Por segurança, a saída do LLM só pode **acrescentar** exclusões alimentares e alergias. Remover uma restrição, declarar “sem alergias” ou reiniciar o pedido só acontece quando a análise local reconhece um pedido explícito do usuário.

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, ConfigDict, Field
from langchain_core.prompts import ChatPromptTemplate

class Interpretacao(BaseModel):
    model_config = ConfigDict(allow_inf_nan=False)
    intencao: Literal["recomendar", "adicionar", "remover", "ver_pedido", "pergunta", "reiniciar"] = "recomendar"
    consulta: str = Field(default="", description="Termos do produto procurado; preserve nomes e ingredientes mencionados")
    restricoes_informadas: bool = Field(default=False, description="True se o usuário expressou preferência ou restrição alimentar/nutricional")
    evita_carne_bovina: bool = False
    evita_frango: bool = False
    evita_porco: bool = False
    evita_peixe: bool = False
    vegetariano: bool = False
    vegano: bool = False
    prefere_menos_sodio: Optional[bool] = None
    prefere_menos_calorias: Optional[bool] = None
    prefere_menos_acucar: Optional[bool] = None
    prefere_mais_proteina: Optional[bool] = None
    max_sodio_mg: Optional[float] = Field(default=None, ge=0)
    max_calorias: Optional[float] = Field(default=None, ge=0)
    max_acucar_g: Optional[float] = Field(default=None, ge=0)
    item_referido: Optional[str] = None
    pergunta: Optional[str] = None
    categoria_desejada: Optional[str] = None  # lanche, sobremesa, bebida, acompanhamento, cafe
    subtipo_lanche: Optional[str] = None  # frango, bovino, porco, peixe, nuggets, outros
    alergias: list[str] = Field(default_factory=list)
    intolerancias: list[str] = Field(default_factory=list)
    sem_alergias: bool = False

prompt_interpretacao = ChatPromptTemplate.from_messages([
    ("system", """Você interpreta mensagens em português para um assistente de cardápio.
Classifique a intenção e extraia somente preferências ou limites expressos pelo usuário.
Não invente produtos, ingredientes, alergênicos, números ou propriedades nutricionais.
'baixo sódio' = prefere_menos_sodio=true, SEM definir um limite numérico.
Se o usuário disser que não se importa mais com um critério, use false nesse campo; se não mencionar, deixe null.
'não como carne de vaca/boi', 'sem carne bovina' = evita_carne_bovina=true.
Palavras dentro de uma negação são restrições, nunca preferência: em 'não como carne', NÃO preencha subtipo_lanche.
Se o usuário disser apenas 'quero um hambúrguer', restricoes_informadas=false.
Se disser 'sem restrições', restricoes_informadas=true.
Para adicionar/remover, identifique item_referido quando possível (ID ou nome exato).
Extraia categoria_desejada (lanche, sobremesa, bebida, acompanhamento, cafe) e subtipo_lanche (frango, bovino, porco, peixe, nuggets) quando pedidos de forma afirmativa.
Extraia alergias declaradas (leite, ovo, trigo/glúten, soja, amendoim, castanhas, peixe, crustáceos, gergelim etc.) e intolerâncias (ex.: lactose).
NÃO infira alergia a partir de preferências ou de restrições nutricionais.
Se a pessoa disser explicitamente que não tem alergias, marque sem_alergias=true.
Perguntas sobre categorias/cardápio podem ter intencao='pergunta'; preserve a consulta.
A consulta deve reter palavras do usuário; não traduza automaticamente nomes comerciais."""),
    ("human", "{mensagem}")
])

def criar_interpretador(modelo_llm):
    """Encadeia prompt + modelo com saída estruturada (Interpretacao)."""
    return prompt_interpretacao | modelo_llm.with_structured_output(Interpretacao)

interpretador = criar_interpretador(globals()["modelo"]) if globals().get("modelo") is not None else None

def interpretar_llm(mensagem):
    """Chama o LLM. Levanta exceção se o modelo não estiver configurado ou falhar."""
    if interpretador is None:
        raise RuntimeError("Modelo não configurado: execute as células de chave e modelo.")
    resultado = interpretador.invoke({"mensagem": mensagem})
    return resultado if isinstance(resultado, Interpretacao) else Interpretacao.model_validate(resultado)

In [ ]:
# @colab-only: exemplo com chamada real ao modelo (consome créditos).
try:
    print(interpretar_llm("Quero frango com pouco sódio, até 500 mg").model_dump())
except Exception as erro:
    print("Exemplo não executado:", type(erro).__name__)

3. Regras determinísticas sobre o CSV

O CSV **não** tem lista de ingredientes nem de alergênicos. Para aplicar restrições sem inventar composição, cada produto recebe, para cada proteína (bovina, frango, suína, peixe), um destes estados:

| estado | significado | efeito sob exclusão |
|---|---|---|
| `indicado` | o nome indica a proteína (ex.: *Hamburger*, *Big Mac*, *Chicken*, *Bacon*, *Fish*) | bloqueado |
| `possivel` | a categoria é compartilhada (*Beef & Pork*, *Chicken & Fish*) ou é um sanduíche sem proteína no nome | bloqueado (conservador) |
| `ausencia_indicada` | o nome diz explicitamente que não tem (ex.: *without Chicken*) | permitido, com aviso |
| `nao_indicado` | nada no nome/categoria indica a proteína | permitido, **com aviso de composição não verificada** |

O léxico de indicação por nome é curado manualmente, foi revisado contra os 260 nomes do CSV e serve apenas para **excluir** (direção conservadora). A **ausência** de um termo no nome nunca é tratada como prova de que o item não contém o ingrediente.

Alergênicos seguem a mesma lógica: um nome que indica o alergênico (ex.: *Cheese*, *Egg*, *Peanut*) bloqueia o item, mas a falta de indicação **não** torna o item seguro.

A função `avaliar_item` centraliza a regra e é usada pela recomendação, pelo ranking e por toda adição ao pedido (ID, nome ou posição).

In [ ]:
import hashlib
import operator

PROTEINAS = ("bovino", "frango", "porco", "peixe")
NOME_PROTEINA = {"bovino": "carne bovina", "frango": "frango", "porco": "carne suína", "peixe": "peixe"}

# Léxico de indicação pelo NOME (normalizado). Usado só para excluir, nunca para garantir ausência.
INDICADORES_PROTEINA = {
    "bovino": r"\b(?:beef|angus|steak|burger|hamburger|cheeseburger|big mac|quarter pounder|mcdouble|daily double|jalapeno double)\b",
    "frango": r"\b(?:chicken|mcchicken|nuggets?|mcnuggets?)\b",
    "porco": r"\b(?:pork|bacon|sausage|ham|mcrib)\b",
    "peixe": r"\b(?:fish|filet-o-fish)\b",
}
AUSENCIA_INDICADA = {"frango": r"\bwithout chicken\b"}
CATEGORIA_PROTEINAS = {"beef & pork": ("bovino", "porco"), "chicken & fish": ("frango", "peixe")}
# Sanduíches cujo recheio não é descrito no nome: proteína não pode ser descartada.
TIPO_SANDUICHE = r"\b(?:sandwich|burger|mcmuffin|biscuit|mcgriddles|bagel|burrito|wrap|mcwrap|delight|breakfast)\b"

INDICADORES_ALERGENO = {
    "leite": r"\b(?:milk|cheese|cheeseburger|cheddar|latte|mocha|frappe|shake|mcflurry|sundae|ice cream|cone|yogurt|parfait|hot chocolate|cream)\b",
    "ovo": r"\b(?:egg|eggs)\b",
    "trigo": r"\b(?:biscuit|bagel|mcmuffin|mcgriddles|hotcakes|cookie|cookies|pie|sandwich|burger|hamburger|cheeseburger|big mac|quarter pounder|mcdouble|mcchicken|mcwrap|wrap|burrito|oreo|nuggets?|mcnuggets?)\b",
    "amendoim": r"\b(?:peanut|reese'?s)\b",
    "castanhas": r"\b(?:hazelnut|almond|pecan|walnut)\b",
    "peixe": r"\b(?:fish|filet-o-fish)\b",
}
INDICADOR_MEL = r"\bhoney\b"

def grupo_produto(categoria, item):
    cat, nome = normalizar(categoria), normalizar(item)
    if any(x in cat for x in ["dessert", "sweet", "sobremesa", "shake", "smoothie"]): return "sobremesa"
    if any(x in cat for x in ["beverage", "drink", "coffee", "tea", "bebida"]): return "bebida"
    if any(x in cat for x in ["breakfast", "cafe da manha"]): return "cafe"
    if any(x in cat for x in ["side", "salad", "snack", "acompanhamento"]): return "acompanhamento"
    if any(x in cat for x in ["beef", "pork", "chicken", "fish", "sandwich", "burger", "lanche"]): return "lanche"
    if re.search(r"nugget|burger|sandwich|mcchicken|filet-o-fish", nome): return "lanche"
    return "outros"

def composicao_indicada(categoria, item, grupo):
    """Estado de cada proteína segundo nome/categoria: indicado, possivel, ausencia_indicada ou nao_indicado."""
    nome, cat = normalizar(item), normalizar(categoria)
    estado = {}
    for p, padrao in INDICADORES_PROTEINA.items():
        ausencia = AUSENCIA_INDICADA.get(p)
        if ausencia and re.search(ausencia, nome):
            estado[p] = "indicado" if re.search(padrao, re.sub(ausencia, " ", nome)) else "ausencia_indicada"
        else:
            estado[p] = "indicado" if re.search(padrao, nome) else None
    alguma = any(v == "indicado" for v in estado.values())
    sanduiche = grupo == "lanche" or re.search(TIPO_SANDUICHE, nome) is not None
    for p in PROTEINAS:
        if estado[p] is None:
            if p in CATEGORIA_PROTEINAS.get(cat, ()) or (sanduiche and not alguma):
                estado[p] = "possivel"
            else:
                estado[p] = "nao_indicado"
    return estado

def alergenos_indicados(item):
    nome = normalizar(item)
    return sorted(a for a, padrao in INDICADORES_ALERGENO.items() if re.search(padrao, nome))

def subtipo_produto(grupo, composicao, item):
    """Divisão de navegação dos lanches, derivada só do NOME (não da categoria compartilhada)."""
    if grupo != "lanche":
        return "nao_aplicavel"
    if re.search(r"nugget", normalizar(item)):
        return "nuggets"
    for p in ("peixe", "frango", "bovino", "porco"):
        if composicao[p] == "indicado":
            return p
    return "outros_lanches"

@dataclass
class BaseMenu:
    df: pd.DataFrame
    documentos: list
    indice: IndiceBusca
    assinatura: str

def verificar_base(base):
    """Garante que DataFrame, documentos e matriz TF-IDF descrevem as mesmas linhas, na mesma ordem."""
    n = len(base.df)
    problemas = []
    if base.df["id_menu"].tolist() != list(range(n)):
        problemas.append("id_menu não corresponde à posição da linha")
    if [d.metadata["id_menu"] for d in base.documentos] != list(range(n)) or \
       [d.metadata["item"] for d in base.documentos] != base.df["item"].tolist():
        problemas.append("documentos não correspondem às linhas")
    if base.indice.matriz.shape[0] != n or list(base.indice.ids) != list(range(n)):
        problemas.append("matriz TF-IDF não corresponde às linhas")
    if problemas:
        raise ValueError("Base desalinhada: " + "; ".join(problemas) +
                         ". Reexecute as células em ordem (ou chame construir_base) e recrie o assistente.")

def construir_base(bruto):
    """Prepara tabela, classificação, documentos e índice em um único passo consistente."""
    tabela = preparar_menu(bruto)
    tabela["grupo"] = [grupo_produto(c, i) for c, i in zip(tabela["categoria"], tabela["item"])]
    composicoes = [composicao_indicada(c, i, g) for c, i, g in zip(tabela["categoria"], tabela["item"], tabela["grupo"])]
    for p in PROTEINAS:
        tabela[f"proteina_{p}"] = [c[p] for c in composicoes]
    tabela["alergenos_nome"] = [alergenos_indicados(i) for i in tabela["item"]]
    tabela["subtipo"] = [subtipo_produto(g, c, i) for g, c, i in zip(tabela["grupo"], composicoes, tabela["item"])]
    tabela["nome_norm"] = tabela["item"].map(normalizar)
    docs = criar_documentos(tabela)
    idx = construir_indice(tabela, docs)
    assinatura = hashlib.sha256("\n".join(tabela["item"] + "|" + tabela["categoria"]).encode()).hexdigest()[:12]
    base = BaseMenu(tabela, docs, idx, assinatura)
    verificar_base(base)
    return base

base = construir_base(df_bruto)
df = base.df  # atalho somente para exibição; o assistente usa a própria base

def formatar_item(r):
    def valor(chave, unidade):
        x = r.get(chave)
        return "não informado" if x is None or pd.isna(x) else f"{x:g} {unidade}"
    return (f"[ID {int(r['id_menu'])}] {r['item']} — {r['categoria']}; "
            f"{valor('calorias', 'kcal')}; sódio {valor('sodio_mg', 'mg')}; "
            f"açúcar {valor('acucar_g', 'g')}; proteína {valor('proteina_g', 'g')}")

print("Base:", len(df), "itens; assinatura", base.assinatura)
print("Categorias originais:", sorted(df["categoria"].unique().tolist()))
display(df.groupby(["grupo", "subtipo"]).size().rename("quantidade").reset_index())
print("Estado da proteína bovina nos lanches:")
display(df[df["grupo"] == "lanche"].groupby("proteina_bovino").size().rename("quantidade").reset_index())

Organização do cardápio e análise local da mensagem

As categorias originais do CSV são preservadas. Os grupos de navegação (lanche, sobremesa, bebida, acompanhamento, café da manhã) e os subtipos de lanche (frango, bovino, porco, peixe, nuggets) vêm do nome e da categoria, para **navegação**. Eles não comprovam composição.

A função `analisar_mensagem` interpreta, sem o LLM, o que pode ser decidido por regras: negações (“não como carne de vaca”), restrições exclusivas (“só como frango”), revogações (“pode incluir carne bovina”), alergias e intolerâncias, limites numéricos com operador e unidade, referências a itens (ID, nome, “o segundo”) e comandos locais. Uma palavra dentro de uma negação **nunca** vira preferência positiva.

In [ ]:
from dataclasses import field

STOP_PT = set("""a o as os um uma uns umas de do da dos das no na nos nas em para pra pro por com e ou que eu me
mim meu minha voce vc quero queria gostaria vou querer quer mostra mostre mostrar ver veja algo alguma algum
alguns algumas opcao opcoes tem ter tipo tipos favor porfavor agora entao tambem so apenas somente mais menos
muito pouco pode poderia sugere sugira sugestao recomenda recomende recomendacao indica indique outro outra
outros outras qual quais coisa coisas cardapio menu mc mcdonalds mcdonald pedido isso esse essa este esta
aqui ai la bom boa gostoso gostosa legal oi ola obrigado obrigada valeu tudo bem como comer como estou ta ser
""".split())
PALAVRAS_GRUPO = set("lanche lanches sanduiche sanduiches sanduba hamburguer hamburgueres burger burgers sobremesa sobremesas doce doces bebida bebidas acompanhamento acompanhamentos cafe manha desjejum".split())
EXPANSAO_PT_EN = {
    "frango": "chicken", "galinha": "chicken", "peixe": "fish", "batata": "fries", "batatas": "fries",
    "fritas": "fries", "refrigerante": "coca cola sprite pepper coke", "refri": "coca cola sprite coke",
    "suco": "juice", "sucos": "juice", "agua": "water", "cafe": "coffee", "cha": "tea", "leite": "milk",
    "sorvete": "sundae cone mcflurry", "casquinha": "cone", "milkshake": "shake", "torta": "pie",
    "maca": "apple", "biscoito": "cookie", "queijo": "cheese", "ovo": "egg", "salada": "salad",
    "saladas": "salad", "morango": "strawberry", "baunilha": "vanilla", "caramelo": "caramel",
    "panqueca": "hotcakes", "panquecas": "hotcakes", "chocolate": "chocolate", "bacon": "bacon",
    "linguica": "sausage", "porco": "pork mcrib", "bife": "steak", "carne": "beef", "nuggets": "mcnuggets nuggets",
    "nugget": "mcnuggets nuggets", "laranja": "orange", "mel": "honey", "mostarda": "mustard", "aveia": "oatmeal",
}
GRUPOS_PT = [
    (r"\bcafe da manha\b|\bdesjejum\b|\bbreakfast\b", "cafe"),
    (r"\blanches?\b|\bsanduiches?\b|\bsanduba\b|\bhamburgu?er(?:es)?\b|\bburgers?\b", "lanche"),
    (r"\bsobremesas?\b|\bdoces?\b|\bsorvetes?\b|\bmilk ?shakes?\b|\bshakes?\b|\btortas?\b|\bsundae\b|\bmcflurry\b|\bcasquinhas?\b", "sobremesa"),
    (r"\bbebidas?\b|\brefrigerantes?\b|\brefri\b|\bsucos?\b|\bagua\b|\bcha\b|\bcafes?\b|\bdrinks?\b", "bebida"),
    (r"\bacompanhamentos?\b|\bbatatas?\b|\bfritas\b|\bsaladas?\b", "acompanhamento"),
]
NOME_GRUPO = {"lanche": "lanches", "sobremesa": "sobremesas", "bebida": "bebidas",
              "acompanhamento": "acompanhamentos", "cafe": "café da manhã"}
TERMOS_PROTEINA_PT = {
    "bovino": r"\bcarnes? (?:de )?(?:vaca|boi|bovinas?)\b|\bbovin[oa]s?\b|\bboi\b|\bvaca\b|\bbeef\b|\bbife\b|\bpicanha\b",
    "frango": r"\bfrangos?\b|\bgalinha\b|\bchicken\b",
    "porco": r"\bcarne (?:de )?porco\b|\bporcos?\b|\bsuin[oa]s?\b|\bbacon\b|\blinguica\b|\bpresunto\b|\bpork\b",
    "peixe": r"\bpeixes?\b|\bfish\b|\bfrutos do mar\b",
}
CARNE_VERMELHA = r"\bcarnes? vermelhas?\b"
PALAVRAS_RESTRICAO = set("""vaca boi bovina bovinas bovino bovinos suino suina suinos suinas porco porcos carnes vermelha
vermelhas galinha frangos peixes sal sodio acucar acucares proteina proteinas caloria calorias kcal gordura gorduras
colesterol carboidrato carboidratos sem nao nunca evito alergia alergias alergico alergica gluten lactose origem animal
restricao restricoes limite limites vegano vegana vegetariano vegetariana hoje""".split())
CARNE_GENERICA = r"\bcarnes?\b"
NEGACAO = r"\b(?:nao|nunca|sem|evito|evitar|evita|nada de|zero|exceto|tirando|tira|tirar)\b"
EXCLUSIVO = r"\b(?:so|somente|apenas|unicamente)\b(?: (?:como|comer|posso comer|quero|aceito))?"
REVOGACAO = (r"\b(?:pode(?:m)? (?:incluir|ter|aparecer|mostrar|voltar)|agora (?:eu )?(?:como|posso)|"
             r"ja (?:como|posso)|voltei a comer|volto a comer|de novo|novamente|liberad[oa]|"
             r"nao tenho mais restric\w*|(?:tirar?|remover?|retirar?|esquec\w*|ignor\w*) (?:a |minha )?restric\w*)\b")
LIMPAR_TUDO = r"\b(?:remov\w*|tir\w*|limp\w*|esquec\w*|ignor\w*|apag\w*) (?:todas )?(?:as |minhas )?restricoes\b|\bnao tenho mais (?:nenhuma )?restric\w*\b"
SEM_RESTRICOES = r"\b(?:sem|nenhuma|nao tenho|nao possuo|zero) restric\w*\b"
SINONIMOS_ALERGENO = {
    "leite": "leite", "lactose": "leite", "laticinios": "leite", "lacteos": "leite", "derivados de leite": "leite",
    "ovo": "ovo", "ovos": "ovo", "trigo": "trigo", "gluten": "trigo", "amendoim": "amendoim",
    "amendoins": "amendoim", "castanha": "castanhas", "castanhas": "castanhas", "nozes": "castanhas",
    "avela": "castanhas", "oleaginosas": "castanhas", "soja": "soja", "peixe": "peixe", "peixes": "peixe",
    "crustaceos": "crustaceos", "camarao": "crustaceos", "frutos do mar": "crustaceos", "gergelim": "gergelim",
    "mostarda": "mostarda",
}
CAMPOS_NUTRI = {
    "calorias": (r"\bcalorias?\b|\bkcal\b|\bcal\b", "calorias", "kcal"),
    "sodio_mg": (r"\bsodio\b|\bsal\b", "sódio", "mg"),
    "acucar_g": (r"\bacucar(?:es)?\b", "açúcar", "g"),
    "proteina_g": (r"\bproteinas?\b", "proteína", "g"),
    "gordura_g": (r"\bgorduras?\b(?! saturadas?)", "gordura", "g"),
    "gordura_sat_g": (r"\bgorduras? saturadas?\b", "gordura saturada", "g"),
    "colesterol_mg": (r"\bcolesterol\b", "colesterol", "mg"),
    "carbo_g": (r"\bcarboidratos?\b|\bcarbos?\b", "carboidratos", "g"),
    "calcio_vd_pct": (r"\bcalcio\b", "cálcio (% VD)", "%"),
}
CAMPOS_LIMITE = ("calorias", "sodio_mg", "acucar_g", "proteina_g", "gordura_g")
PADRAO_LIMITE = re.compile(
    r"(?P<op>nao mais (?:de|que|do que)|no maximo|maximo de|pelo menos|no minimo|minimo de|menos de|menos que|"
    r"abaixo de|inferior a|menor que|mais de|mais que|acima de|superior a|maior que|ate|<=|>=|<|>)\s*"
    r"(?P<num>-?\s*\d+(?:[.,]\d+)?)(?P<resto>[^.;!?\n]{0,40})")
OPERADOR = {"nao mais de": "<=", "nao mais que": "<=", "nao mais do que": "<=", "no maximo": "<=",
            "maximo de": "<=", "ate": "<=", "<=": "<=", "menos de": "<", "menos que": "<", "abaixo de": "<",
            "inferior a": "<", "menor que": "<", "<": "<", "pelo menos": ">=", "no minimo": ">=",
            "minimo de": ">=", ">=": ">=", "mais de": ">", "mais que": ">", "acima de": ">",
            "superior a": ">", "maior que": ">", ">": ">"}
PREFERENCIAS_REL = {
    "prefere_menos_sodio": (r"\b(?:pouco|menos|baixo|reduzido|baixa quantidade de) (?:teor de )?(?:sodio|sal)\b", "sodio_mg"),
    "prefere_menos_calorias": (r"\b(?:menos|poucas|baixas?) calorias?\b|\bmais leves?\b|\bbaixa caloria\b|\blight\b", "calorias"),
    "prefere_menos_acucar": (r"\b(?:menos|pouco|baixo) acucar\b", "acucar_g"),
    "prefere_mais_proteina": (r"\b(?:mais|muita|alta|rico em|rica em) proteinas?\b", "proteina_g"),
}
ORDINAIS = {"primeiro": 1, "primeira": 1, "segundo": 2, "segunda": 2, "terceiro": 3, "terceira": 3,
            "quarto": 4, "quarta": 4, "quinto": 5, "quinta": 5}

@dataclass
class AnaliseLocal:
    """Resultado determinístico da leitura de uma mensagem (sem LLM)."""
    reiniciar: bool = False
    ver_pedido: bool = False
    comando: Optional[str] = None             # 'adicionar' | 'remover'
    id_ref: Optional[int] = None
    ordinal: Optional[int] = None             # 1-based; -1 = último
    nome_ref: Optional[int] = None
    nomes_ambiguos: list = field(default_factory=list)
    numero_solto: Optional[str] = None
    erro_referencia: Optional[str] = None
    confirmacao: bool = False
    excluir: set = field(default_factory=set)
    liberar: set = field(default_factory=set)
    dieta: Optional[str] = None               # 'vegetariana' | 'vegana' | 'nenhuma'
    carne_generica_negada: bool = False
    limpar_restricoes: bool = False
    sem_restricoes: bool = False
    resposta_negativa: bool = False
    alergias: set = field(default_factory=set)
    intolerancias: set = field(default_factory=set)
    sem_alergias: bool = False
    alergia_nao_especificada: bool = False
    preferencias: dict = field(default_factory=dict)
    limites: dict = field(default_factory=dict)      # campo -> (op, valor)
    remover_limites: set = field(default_factory=set)
    limite_ambiguo: list = field(default_factory=list)
    limite_invalido: list = field(default_factory=list)
    grupo: Optional[str] = None
    subtipo: Optional[str] = None
    revoga_subtipo: bool = False
    campos_pergunta: list = field(default_factory=list)
    pergunta_composicao: bool = False
    termos_busca: list = field(default_factory=list)

    @property
    def informou_restricao(self):
        return bool(self.excluir or self.liberar or self.dieta or self.limpar_restricoes or self.sem_restricoes
                    or self.resposta_negativa or self.alergias or self.intolerancias or self.sem_alergias
                    or self.alergia_nao_especificada or self.preferencias or self.limites or self.remover_limites
                    or self.carne_generica_negada)

def _clausulas(t):
    return [c.strip() for c in re.split(r"[.,;!?\n]|\bmas\b|\bporem\b|\bcontudo\b|\bentretanto\b", t) if c.strip()]

def _proteinas_em(texto):
    achadas = {p for p, padrao in TERMOS_PROTEINA_PT.items() if re.search(padrao, texto)}
    if re.search(CARNE_VERMELHA, texto):
        achadas |= {"bovino", "porco"}
    return achadas

def _alergenos_em(texto):
    achados = set()
    for termo, alergeno in sorted(SINONIMOS_ALERGENO.items(), key=lambda x: -len(x[0])):
        if re.search(r"\b" + re.escape(termo) + r"\b", texto):
            achados.add(alergeno)
    return achados

def _nutriente_em(texto, de_tras_para_frente=False):
    achados = []
    for campo, (padrao, _, _) in CAMPOS_NUTRI.items():
        for m in re.finditer(padrao, texto):
            achados.append((m.start(), campo))
    if not achados:
        return None
    achados.sort()
    return achados[-1][1] if de_tras_para_frente else achados[0][1]

def _ler_limites(t, a):
    for m in PADRAO_LIMITE.finditer(t):
        op = OPERADOR[re.sub(r"\s+", " ", m.group("op"))]
        bruto = m.group("num").replace(" ", "")
        resto = m.group("resto")
        if re.match(r"\s*(?:opcoes|itens|unidades|pecas|pedacos|reais|dolares|anos|minutos|pessoas)\b", resto):
            continue
        unidade_m = re.match(r"\s*(kcal|cal|calorias?|mg|miligramas?|g|gramas?)\b", resto)
        unidade = unidade_m.group(1) if unidade_m else None
        campo = _nutriente_em(resto) or _nutriente_em(t[max(0, m.start() - 40):m.start()], de_tras_para_frente=True)
        if campo is None and unidade in ("kcal", "cal", "caloria", "calorias"):
            campo = "calorias"
        if campo is None and unidade in ("mg", "miligrama", "miligramas"):
            campo = "sodio_mg"
        if bruto.startswith("-"):
            a.limite_invalido.append(f"{m.group('op')} {bruto}: valores negativos não são aceitos")
            continue
        valor = float(bruto.replace(",", "."))
        if campo is None or campo not in CAMPOS_LIMITE:
            a.limite_ambiguo.append((op, valor))
            continue
        if unidade in ("g", "grama", "gramas") and campo == "sodio_mg":
            valor *= 1000
        elif unidade in ("mg", "miligrama", "miligramas") and campo in ("acucar_g", "proteina_g", "gordura_g"):
            valor /= 1000
        elif unidade in ("mg", "miligrama", "miligramas", "g", "grama", "gramas") and campo == "calorias":
            a.limite_invalido.append(f"{valor:g} {unidade} não é unidade de energia (use kcal)")
            continue
        a.limites[campo] = (op, valor)

def _resolver_nome(t, base):
    """Procura nomes de itens do CSV dentro do texto (maior nome primeiro)."""
    tabela = base.df
    exatos = [(len(n), i) for i, n in zip(tabela["id_menu"], tabela["nome_norm"])
              if re.search(r"(?<![\w-])" + re.escape(n) + r"(?![\w-])", t)]
    if exatos:
        maior = max(exatos)[0]
        ids = [i for n, i in exatos if n == maior]
        return (ids[0], []) if len(ids) == 1 else (None, ids)
    bases = {}
    for i, n in zip(tabela["id_menu"], tabela["nome_norm"]):
        raiz = re.sub(r"\s*\(.*?\)\s*", " ", n).strip()
        if len(raiz) >= 3 and re.search(r"(?<![\w-])" + re.escape(raiz) + r"(?![\w-])", t):
            bases.setdefault(raiz, []).append(int(i))
    if bases:
        maior = max(bases, key=len)
        ids = bases[maior]
        return (ids[0], []) if len(ids) == 1 else (None, ids)
    return None, []

def analisar_mensagem(texto, base):
    t = normalizar(texto)
    a = AnaliseLocal()
    if not t:
        return a
    # Comandos de estado
    if re.search(r"\b(?:reinicia\w*|recomeca\w*|recomecar|zera\w*|novo pedido|cancela\w* (?:o |meu )?pedido|limpa\w* (?:o |meu )?pedido)\b", t) \
            and not re.search(r"restric", t):
        a.reiniciar = True
        return a
    if re.search(r"\b(?:ver|mostra\w*|exib\w*|consulta\w*|como (?:esta|ficou)|qual (?:e|eh))\b.{0,12}\b(?:meu |o )?pedido\b", t) \
            or t in ("pedido", "meu pedido", "carrinho", "ver pedido") or re.search(r"\bo que (?:tem|esta) no (?:meu )?pedido\b", t):
        a.ver_pedido = True
    a.confirmacao = bool(re.search(r"\b(?:estou ciente|ciente|confirmo|aceito o risco)\b", t))
    a.resposta_negativa = t in ("nao", "nenhuma", "nenhum", "nada", "nao tenho", "nenhuma restricao", "sem")

    # Referências a itens
    m = re.search(r"\bid\s*[:#]?\s*(-?\s*\d+(?:[.,]\d+)?)", t)
    if m:
        bruto = m.group(1).replace(" ", "")
        if bruto.startswith("-") or re.search(r"[.,]", bruto):
            a.erro_referencia = f"'{bruto}' não é um ID válido: use um número inteiro entre 0 e {len(base.df) - 1}."
        else:
            n = int(bruto)
            if n >= len(base.df):
                a.erro_referencia = f"ID {n} não existe no menu.csv (IDs vão de 0 a {len(base.df) - 1})."
            else:
                a.id_ref = n
    if a.id_ref is None and not a.erro_referencia:
        m = re.search(r"\b(primeir[oa]|segund[oa]|terceir[oa]|quart[oa]|quint[oa])\b", t)
        if m:
            a.ordinal = ORDINAIS[m.group(1)]
        elif re.search(r"\bultim[oa]\b", t):
            a.ordinal = -1
        else:
            m = re.search(r"\b(\d+)(?:o|a)\b|\b(?:opcao|numero|n|item|posicao)\s*(\d+)\b", t)
            if m:
                a.ordinal = int(m.group(1) or m.group(2))
    if a.id_ref is None and a.ordinal is None and not a.erro_referencia:
        a.nome_ref, a.nomes_ambiguos = _resolver_nome(t, base)

    # Restrições por cláusula (a negação vale só dentro da sua cláusula)
    a.limpar_restricoes = bool(re.search(LIMPAR_TUDO, t))
    a.sem_restricoes = bool(re.search(SEM_RESTRICOES, t)) and not a.limpar_restricoes
    positivas = []
    for c in _clausulas(t):
        if re.search(r"\b(?:alergi\w*|alergic\w*|intoleran\w*|celiac\w*|doenca celiaca)\b", c):
            neg = re.search(r"\b(?:nao|sem|nenhuma?|zero)\b", c)
            alvo = _alergenos_em(c)
            intolerancia = re.search(r"intoleran", c)
            if re.search(r"celiac", c):
                a.alergias.add("trigo")
            elif neg and not alvo:
                a.sem_alergias = True
            elif not neg:
                if alvo:
                    (a.intolerancias if intolerancia else a.alergias).update(alvo)
                else:
                    resto = re.split(r"\b(?:alergi\w*|alergic\w*|intoleran\w*)\b", c, maxsplit=1)[-1]
                    resto = re.sub(r"^\s*(?:a|ao|as|aos|de|com)\b", "", resto).strip()
                    if resto:
                        (a.intolerancias if intolerancia else a.alergias).add(resto)
                    else:
                        a.alergia_nao_especificada = True
            continue
        if re.search(r"\b(?:nao sou|deixei de ser|nao sou mais)\b.{0,6}\b(?:vegetarian|vegan)\w*", c):
            a.dieta = "nenhuma"
            continue
        if re.search(r"\bvegan[oa]s?\b", c) or (re.search(r"\borigem animal\b", c) and re.search(NEGACAO, c)):
            a.dieta = "vegana"
            continue
        if re.search(r"\bvegetarian[oa]s?\b", c):
            a.dieta = "vegetariana" if a.dieta != "vegana" else a.dieta
            continue
        if re.search(r"\b(?:restric\w*|limites?)\b", c) and not _proteinas_em(c):
            continue
        ingrediente = re.search(r"\b(?:sem|nao (?:posso|como|tomo|consumo)|evito)\b[^.]*?\b(gluten|lactose|leite|ovos?|amendoins?|trigo|soja|castanhas?|nozes)\b", c)
        if ingrediente and not _proteinas_em(c):
            a.intolerancias |= _alergenos_em(ingrediente.group(1))
            positivas.append(c[:ingrediente.start()])
            continue
        prot = _proteinas_em(c)
        generica = re.search(CARNE_GENERICA, c) and not prot
        revoga = re.search(REVOGACAO, c)
        negada = re.search(NEGACAO, c)
        exclusivo = re.search(EXCLUSIVO, c)
        if re.search(r"\b(?:nao precisa ser|tanto faz|qualquer)\b", c) and (prot or re.search(r"\btipo\b", c)):
            a.revoga_subtipo = True
            continue
        if prot or generica:
            if revoga and not re.search(r"\bnao (?:como|quero|posso)\b", c):
                a.liberar |= prot or {"bovino", "porco"}
            elif negada:
                positivas.append(c[:negada.start()])
                if generica:
                    a.excluir |= {"bovino", "porco"}
                    a.carne_generica_negada = True
                else:
                    a.excluir |= prot
            elif exclusivo and prot:
                a.excluir |= set(PROTEINAS) - prot
                if len(prot) == 1:
                    a.subtipo = next(iter(prot))
            else:
                if len(prot) == 1:
                    a.subtipo = next(iter(prot))
                elif generica:
                    a.subtipo = "bovino"
                positivas.append(c)
            continue
        positivas.append(c[:negada.start()] if negada else c)

    # Nuggets como subtipo (positivo)
    texto_pos = " ".join(positivas)
    if re.search(r"\b(?:mc)?nuggets?\b", texto_pos):
        a.subtipo = "nuggets"
    # Grupo (só em cláusulas afirmativas)
    sem_cafe_manha = texto_pos
    for padrao, grupo in GRUPOS_PT:
        if re.search(padrao, sem_cafe_manha):
            a.grupo = grupo
            break

    # Limites, preferências e revogações
    _ler_limites(t, a)
    for campo_pref, (padrao, _) in PREFERENCIAS_REL.items():
        if re.search(padrao, t):
            a.preferencias[campo_pref] = True
    if re.search(r"\bsem acucar\b", t):
        a.limites["acucar_g"] = ("<=", 0.0)
    for campo_pref, (_, coluna) in PREFERENCIAS_REL.items():
        padrao_nutri = CAMPOS_NUTRI[coluna][0]
        if re.search(r"\b(?:nao (?:me )?import\w*|tanto faz|nao precisa|pode ter|nao ligo|esquec\w*|ignor\w*|sem limite)\b[^.;,]*(?:" + padrao_nutri + ")", t):
            a.preferencias[campo_pref] = False
            a.remover_limites.add(coluna)
            a.limites.pop(coluna, None)
    if re.search(r"\b(?:remov\w*|tir\w*|retir\w*|esquec\w*|ignor\w*|sem)\b (?:o |os |todos os )?limites?\b", t):
        alvo = _nutriente_em(t)
        a.remover_limites |= {alvo} if alvo in CAMPOS_LIMITE else set(CAMPOS_LIMITE)

    # Comandos de pedido
    tem_ref = a.id_ref is not None or a.ordinal is not None or a.nome_ref is not None or bool(a.nomes_ambiguos) or bool(a.erro_referencia)
    if not tem_ref:
        m = re.search(r"\b\d+\b", t)
        if m and not a.limites and not a.limite_ambiguo and not a.limite_invalido:
            a.numero_solto = m.group(0)
    if re.search(r"\b(?:remov\w*|retir\w*|exclu\w*|apag\w*|delet\w*|tira\w*)\b", t) and (tem_ref or a.numero_solto or re.search(r"\bdo (?:meu )?pedido\b", t)):
        if not (a.remover_limites or a.liberar or a.limpar_restricoes):
            a.comando = "remover"
    elif re.search(r"\b(?:adicion\w*|acrescent\w*|coloc\w*|inclu\w*|bot[ae]\w*|poe|por no pedido|vou querer|manda|pega)\b", t) or \
            re.search(r"\b(?:quero|pode ser|escolho|fico com|me ve|me da)\b (?:o|a|esse|essa|este|esta)\b", t):
        if (tem_ref or a.numero_solto) and not a.liberar:
            a.comando = "adicionar"
        elif re.search(r"\b(?:adicion\w*|acrescent\w*|coloc\w*|bot[ae]\w*|por no pedido)\b", t) and not a.informou_restricao:
            a.comando = "adicionar"

    # Perguntas sobre nutrientes ou composição
    if not a.limites and not a.limite_ambiguo and re.search(r"\b(?:quant[oa]s?|qual|quais|tem|possui|valor|informa\w*)\b", t):
        a.campos_pergunta = [c for c, (padrao, _, _) in CAMPOS_NUTRI.items() if re.search(padrao, t)]
        if re.search(r"\bporcao\b|\btamanho\b|\bpeso\b", t):
            a.campos_pergunta.append("porcao_texto")
    if re.search(r"\b(?:e seguro|eh seguro|seguro para|posso comer|tem (?:gluten|lactose|leite|ovo|amendoim|trigo|soja|castanha|carne|frango|porco|peixe|bacon)|contem|ingredientes?|e vegan|e vegetarian|alergenic\w*)\b", t):
        a.pergunta_composicao = True

    # Termos de busca: só das cláusulas afirmativas, sem palavras genéricas
    tokens = re.findall(r"[a-z0-9&'-]+", texto_pos)
    a.termos_busca = [w for w in tokens if w not in STOP_PT and w not in PALAVRAS_GRUPO and not re.search(r"\d", w)
                      and w not in PALAVRAS_RESTRICAO
                      and w not in ("id", "kcal", "mg", "g", "calorias", "caloria", "sodio", "acucar", "proteina",
                                    "frango", "galinha", "peixe", "carne", "bovina", "bovino", "porco", "nuggets", "nugget",
                                    "segundo", "segunda", "primeiro", "primeira", "terceiro", "terceira", "ultimo", "ultima",
                                    "adicione", "adicionar", "adiciona", "remove", "remover", "remova", "coloca", "coloque",
                                    "da", "manha", "ate", "de", "baixo", "pouco", "leve", "light", "maximo", "minimo")]
    if a.campos_pergunta or a.pergunta_composicao or a.comando or a.ver_pedido or a.nome_ref is not None:
        a.termos_busca = []
    return a

def consulta_para_busca(termos):
    return " ".join(EXPANSAO_PT_EN.get(w, w) for w in termos)

def mostrar_cardapio(grupo=None, subtipo=None, limite=12, tabela=None):
    itens = base.df if tabela is None else tabela
    if grupo: itens = itens[itens["grupo"] == grupo]
    if subtipo: itens = itens[itens["subtipo"] == subtipo]
    if itens.empty: return "Nenhum produto encontrado nessa divisão do CSV."
    linhas = [formatar_item(r) for r in itens.head(limite).to_dict("records")]
    resto = len(itens) - len(linhas)
    return "\n".join(linhas) + (f"\n... e mais {resto} produtos. Refine o tipo ou peça mais opções." if resto else "")

print("Exemplo de lanches de frango:")
print(mostrar_cardapio("lanche", "frango", 5))
print("\nAnálise local de 'eu nao como carne de vaca':", analisar_mensagem("eu nao como carne de vaca", base).excluir)

4. Assistente conversacional com RAG sobre o CSV

O fluxo de cada mensagem é:

1. **Análise local** (`analisar_mensagem`): comandos inequívocos, restrições com negação, limites, referências.
2. **LLM** (`interpretar_llm`): só é chamado quando a mensagem não é um comando local. Se falhar ou estourar o tempo, o assistente continua com a análise local e avisa o usuário. O pedido nunca é alterado por causa de uma falha do modelo.
3. **Estado**: exclusões rígidas, dieta, alergias, intolerâncias, preferências relativas e limites ficam em campos separados. Uma mudança explícita altera só o campo correspondente, invalida a lista anterior de recomendações e reavalia os itens já no pedido.
4. **Recuperação**: `avaliar_item` remove os candidatos incompatíveis **antes** do ranking TF-IDF. Depois, o ranking é ajustado pelas preferências relativas. A lista final é formatada a partir do CSV, sem pedir ao LLM que reescreva valores.

As funções `consultar_menu` e `consultar_item_por_id` (ferramentas LangChain) ficam disponíveis para experimentos. **O fluxo da conversa não as invoca por *tool calling***: ele chama diretamente as mesmas funções de busca e de avaliação.

In [ ]:
from langchain_core.tools import tool

@tool
def consultar_menu(consulta: str) -> str:
    """Consulta produtos do menu.csv por similaridade textual, sem inventar itens."""
    resultados = [(i, s) for i, s in buscar_similares(base.indice, consulta, k=5) if s > 0]
    return "\n".join(formatar_item(base.df.iloc[i].to_dict()) for i, _ in resultados) or "Nenhum item encontrado."

@tool
def consultar_item_por_id(id_menu: int) -> str:
    """Obtém os valores nutricionais reais de um produto por ID no CSV."""
    if isinstance(id_menu, bool) or id_menu < 0 or id_menu >= len(base.df):
        return "ID inexistente no menu.csv."
    return formatar_item(base.df.iloc[id_menu].to_dict())

ferramentas = [consultar_menu, consultar_item_por_id]
print("Ferramentas disponíveis (não usadas no fluxo da conversa):", [t.name for t in ferramentas])

In [ ]:
AVISO_ALERGIA = ("ATENÇÃO: o CSV não informa alergênicos, ingredientes completos nem contaminação cruzada. "
                 "Não é possível afirmar que qualquer item seja seguro. Consulte a tabela oficial atualizada "
                 "e confirme com o restaurante antes de consumir.")
AVISO_COMPOSICAO = ("Compatibilidade avaliada só pelo nome e pela categoria do CSV. Estes itens não têm a "
                    "restrição indicada no nome, mas os ingredientes não foram verificados.")
AVISO_VEGANO = "O CSV não permite confirmar que algum item é vegano (não há lista de ingredientes)."
COMPARA = {"<": operator.lt, "<=": operator.le, ">": operator.gt, ">=": operator.ge}
TEXTO_OP = {"<": "menor que", "<=": "até", ">": "maior que", ">=": "pelo menos"}

@dataclass
class EstadoPedido:
    exclusoes: set = field(default_factory=set)         # proteínas excluídas (regra rígida)
    dieta: Optional[str] = None                          # 'vegetariana' | 'vegana'
    alergias: set = field(default_factory=set)
    intolerancias: set = field(default_factory=set)
    sem_alergias_declaradas: bool = False
    restricoes_respondidas: bool = False
    preferencias: dict = field(default_factory=dict)     # preferências relativas (ordenação)
    limites: dict = field(default_factory=dict)          # campo -> (operador, valor) (filtro rígido)
    categoria_atual: str = ""
    subtipo_atual: str = ""
    pedido: list = field(default_factory=list)
    ultimas_recomendacoes: list = field(default_factory=list)
    motivo_lista_invalida: str = ""
    confirmacao_pendente: Optional[int] = None
    termos_pendentes: list = field(default_factory=list)
    perguntas_restricao: int = 0
    historico: list = field(default_factory=list)

def _rotulo_valor(campo, valor):
    _, rotulo, unidade = CAMPOS_NUTRI[campo]
    return f"{rotulo} {valor:g} {unidade}"

def avaliar_item(base, estado, id_menu):
    """Regra única de compatibilidade. Devolve (bloqueios, avisos). Sem bloqueio NÃO significa 'seguro'."""
    r = base.df.iloc[int(id_menu)]
    bloqueios, avisos = [], []
    exclusoes = set(estado.exclusoes) | (set(PROTEINAS) if estado.dieta in ("vegetariana", "vegana") else set())
    for p in PROTEINAS:
        if p not in exclusoes:
            continue
        situacao = r[f"proteina_{p}"]
        if situacao == "indicado":
            bloqueios.append(f"o nome indica {NOME_PROTEINA[p]}")
        elif situacao == "possivel":
            bloqueios.append(f"não é possível descartar {NOME_PROTEINA[p]} (categoria '{r['categoria']}' ou sanduíche sem recheio descrito)")
    if estado.dieta == "vegana":
        animais = set(r["alergenos_nome"]) & {"leite", "ovo", "peixe"}
        if animais:
            bloqueios.append("o nome indica ingrediente de origem animal (" + ", ".join(sorted(animais)) + ")")
        elif re.search(INDICADOR_MEL, r["nome_norm"]):
            bloqueios.append("o nome indica mel")
        else:
            avisos.append(AVISO_VEGANO)
    for alergeno in sorted((estado.alergias | estado.intolerancias) & set(r["alergenos_nome"])):
        tipo = "alergia" if alergeno in estado.alergias else "intolerância"
        bloqueios.append(f"o nome indica {alergeno} ({tipo} informada)")
    for campo, (op, limite) in estado.limites.items():
        x = r[campo]
        if pd.isna(x):
            bloqueios.append(f"{CAMPOS_NUTRI[campo][1]} não informado no CSV")
        elif not COMPARA[op](float(x), float(limite)):
            bloqueios.append(f"{_rotulo_valor(campo, x)} não atende ao limite ({TEXTO_OP[op]} {limite:g} {CAMPOS_NUTRI[campo][2]})")
    if exclusoes and not bloqueios:
        avisos.append(AVISO_COMPOSICAO)
    return bloqueios, avisos

def _descrever_restricoes(estado):
    partes = []
    if estado.dieta:
        partes.append(f"dieta {estado.dieta}")
    if estado.exclusoes:
        partes.append("sem " + ", ".join(NOME_PROTEINA[p] for p in sorted(estado.exclusoes)))
    for campo, (op, v) in estado.limites.items():
        partes.append(f"{CAMPOS_NUTRI[campo][1]} {TEXTO_OP[op]} {v:g} {CAMPOS_NUTRI[campo][2]}")
    if estado.alergias:
        partes.append("alergia: " + ", ".join(sorted(estado.alergias)))
    if estado.intolerancias:
        partes.append("intolerância/evitar ingrediente: " + ", ".join(sorted(estado.intolerancias)))
    return "; ".join(partes) if partes else "nenhuma"

class AssistenteMcDonalds:
    def __init__(self, base, interpretar_fn=None):
        verificar_base(base)
        self.base = base
        self.interpretar_fn = interpretar_fn
        self.estado = EstadoPedido()

    # ---------- utilidades ----------
    def _item(self, i):
        return self.base.df.iloc[int(i)]

    def _invalidar_lista(self, motivo):
        if self.estado.ultimas_recomendacoes:
            self.estado.motivo_lista_invalida = motivo
        self.estado.ultimas_recomendacoes = []

    def _resolver_referencia(self, a):
        """Devolve (id, erro). Nunca adivinha: referências ambíguas viram pergunta."""
        if a.erro_referencia:
            return None, a.erro_referencia
        if a.id_ref is not None:
            return a.id_ref, None
        if a.ordinal is not None:
            lista = self.estado.ultimas_recomendacoes
            if not lista:
                motivo = self.estado.motivo_lista_invalida
                return None, ("Não há lista de opções ativa" + (f" ({motivo})" if motivo else "") +
                              ". Peça novas recomendações ou informe o ID.")
            pos = len(lista) if a.ordinal == -1 else a.ordinal
            if not 1 <= pos <= len(lista):
                return None, f"A lista atual tem {len(lista)} opções; não existe a opção {a.ordinal}."
            return lista[pos - 1], None
        if a.nome_ref is not None:
            return a.nome_ref, None
        if a.nomes_ambiguos:
            opcoes = "\n".join(formatar_item(self._item(i).to_dict()) for i in a.nomes_ambiguos[:8])
            return None, "Há mais de um item com esse nome. Qual deles? Informe o ID:\n" + opcoes
        if a.numero_solto is not None:
            return None, (f"Você quis dizer o ID {a.numero_solto} ou a opção {a.numero_solto} da última lista? "
                          f"Escreva 'ID {a.numero_solto}' ou 'a opção {a.numero_solto}'.")
        return None, "Qual produto? Informe o ID exibido, a posição na lista (ex.: 'o segundo') ou o nome exato."

    def _conflitos_pedido(self):
        conflitos = []
        for i in dict.fromkeys(self.estado.pedido):
            bloqueios, _ = avaliar_item(self.base, self.estado, i)
            if bloqueios:
                conflitos.append(f"- {self._item(i)['item']} (ID {i}): " + "; ".join(bloqueios))
        if not conflitos:
            return ""
        return ("Atenção: itens já no seu pedido não atendem às restrições atuais:\n" + "\n".join(conflitos) +
                "\nPara retirar, diga por exemplo 'remover ID N'.")

    # ---------- atualização de estado ----------
    def _aplicar_restricoes(self, a, llm=None):
        e = self.estado
        antes = (set(e.exclusoes), e.dieta, set(e.alergias), set(e.intolerancias), dict(e.limites))
        notas = []
        if a.limpar_restricoes:
            e.exclusoes.clear(); e.dieta = None; e.limites.clear(); e.preferencias.clear()
            notas.append("Removi as restrições alimentares e nutricionais (alergias continuam registradas).")
        if a.dieta == "nenhuma":
            e.dieta = None
        elif a.dieta:
            e.dieta = a.dieta
        e.exclusoes |= a.excluir
        e.exclusoes -= (a.liberar - a.excluir)
        if a.liberar and e.dieta in ("vegetariana", "vegana"):
            notas.append(f"Sua dieta {e.dieta} continua registrada; diga 'não sou mais {e.dieta[:-1]}o' para mudar.")
        if a.sem_alergias:
            e.alergias.clear(); e.intolerancias.clear(); e.sem_alergias_declaradas = True
        if a.alergias or a.intolerancias:
            e.alergias |= a.alergias; e.intolerancias |= a.intolerancias; e.sem_alergias_declaradas = False
        for campo in a.remover_limites:
            e.limites.pop(campo, None)
        e.limites.update(a.limites)
        for campo, valor in a.preferencias.items():
            if valor:
                e.preferencias[campo] = True
            else:
                e.preferencias.pop(campo, None)
        if llm is not None:
            notas += self._aplicar_llm(llm, a)
        if a.carne_generica_negada and not ({"frango", "peixe"} & (a.excluir | a.liberar)):
            notas.append("Entendi que você não come carne bovina nem suína. Se também evita frango ou peixe, me avise.")
        depois = (set(e.exclusoes), e.dieta, set(e.alergias), set(e.intolerancias), dict(e.limites))
        mudou = antes != depois
        if mudou:
            e.confirmacao_pendente = None
            self._invalidar_lista("suas restrições mudaram")
            if e.subtipo_atual in e.exclusoes or (e.dieta and e.subtipo_atual in PROTEINAS + ("nuggets",)):
                e.subtipo_atual = ""
        return mudou, notas

    def _aplicar_llm(self, p, a):
        """Saída do LLM só ACRESCENTA restrições; nunca remove exclusões, alergias ou reinicia."""
        e = self.estado
        notas = []
        mapa = {"evita_carne_bovina": "bovino", "evita_frango": "frango", "evita_porco": "porco", "evita_peixe": "peixe"}
        for campo, proteina in mapa.items():
            if getattr(p, campo) and proteina not in a.liberar and proteina != a.subtipo:
                e.exclusoes.add(proteina)
        if p.vegano and a.dieta != "nenhuma":
            e.dieta = "vegana"
        elif p.vegetariano and a.dieta != "nenhuma" and e.dieta != "vegana":
            e.dieta = "vegetariana"
        if not a.sem_alergias:
            for x in p.alergias:
                e.alergias |= _alergenos_em(normalizar(x)) or {normalizar(x)}
            for x in p.intolerancias:
                e.intolerancias |= _alergenos_em(normalizar(x)) or {normalizar(x)}
        for pref in PREFERENCIAS_REL:
            valor = getattr(p, pref)
            if pref in a.preferencias or valor is None:
                continue
            if valor:
                e.preferencias[pref] = True
            else:
                e.preferencias.pop(pref, None)
        for campo_llm, coluna in (("max_sodio_mg", "sodio_mg"), ("max_calorias", "calorias"), ("max_acucar_g", "acucar_g")):
            valor = getattr(p, campo_llm)
            if valor is None or coluna in a.limites or coluna in a.remover_limites:
                continue
            op = next((o for o, v in a.limite_ambiguo if v == valor), "<=")
            e.limites[coluna] = (op, float(valor))
        return notas

    def _atualizar_navegacao(self, a, llm=None):
        e = self.estado
        notas = []
        grupo = a.grupo
        if grupo is None and llm is not None and llm.categoria_desejada:
            grupo = next((g for padrao, g in GRUPOS_PT if re.search(padrao, normalizar(llm.categoria_desejada))), None) \
                or (normalizar(llm.categoria_desejada) if normalizar(llm.categoria_desejada) in NOME_GRUPO else None)
        subtipo = a.subtipo
        if subtipo is None and llm is not None and llm.subtipo_lanche:
            s = normalizar(llm.subtipo_lanche)
            s = {"carne": "bovino", "boi": "bovino", "porco": "porco", "suino": "porco", "nugget": "nuggets"}.get(s, s)
            # O subtipo do LLM é ignorado se a palavra só apareceu negada na mensagem.
            if s in PROTEINAS + ("nuggets",) and s not in a.excluir:
                subtipo = s
        if a.revoga_subtipo:
            e.subtipo_atual = ""
        if grupo and grupo != e.categoria_atual:
            e.categoria_atual = grupo
            e.subtipo_atual = ""
        if subtipo:
            bloqueado = subtipo in e.exclusoes or (e.dieta and subtipo in PROTEINAS + ("nuggets",)) or \
                (subtipo == "nuggets" and "frango" in e.exclusoes)
            if bloqueado:
                nome_p = NOME_PROTEINA.get(subtipo, "frango")
                notas.append(f"Você informou que evita {nome_p}; não vou mostrar esse tipo. "
                             f"Se mudou de ideia, diga 'pode incluir {nome_p}'.")
            else:
                # Nuggets só existem como lanche; sem grupo na mensagem, a navegação vai para lanches.
                if not grupo and (not e.categoria_atual or subtipo == "nuggets"):
                    e.categoria_atual = "lanche"
                e.subtipo_atual = subtipo
        return notas

    # ---------- respostas determinísticas ----------
    def _ver_pedido(self):
        e = self.estado
        if not e.pedido:
            return "Seu pedido está vazio. Quer ver lanches, sobremesas, bebidas, acompanhamentos ou café da manhã?"
        contagem = {}
        for i in e.pedido:
            contagem[i] = contagem.get(i, 0) + 1
        linhas = [f"{n}x " + formatar_item(self._item(i).to_dict()) for i, n in contagem.items()]
        totais = []
        for campo in ("calorias", "sodio_mg", "acucar_g", "proteina_g"):
            valores = [self._item(i)[campo] for i in e.pedido]
            soma = sum(v for v in valores if not pd.isna(v))
            faltam = sum(1 for v in valores if pd.isna(v))
            totais.append(_rotulo_valor(campo, soma) + (f" (+{faltam} sem informação)" if faltam else ""))
        texto = "Seu pedido:\n" + "\n".join(linhas) + "\nTotal (CSV): " + "; ".join(totais)
        conflitos = self._conflitos_pedido()
        if conflitos:
            texto += "\n" + conflitos
        if e.alergias or e.intolerancias:
            texto += "\n" + AVISO_ALERGIA
        return texto

    def _adicionar(self, id_):
        e = self.estado
        r = self._item(id_)
        bloqueios, avisos = avaliar_item(self.base, e, id_)
        if bloqueios:
            e.confirmacao_pendente = None
            return (f"Não adicionei {r['item']} (ID {id_}): " + "; ".join(bloqueios) +
                    f". Restrições atuais: {_descrever_restricoes(e)}.")
        if e.alergias and e.confirmacao_pendente != id_:
            e.confirmacao_pendente = id_
            return (f"Antes de adicionar {r['item']} (ID {id_}): você informou alergia a "
                    + ", ".join(sorted(e.alergias)) + ". " + AVISO_ALERGIA +
                    f"\nSe mesmo assim quiser registrar este item no pedido, escreva: 'confirmo adicionar ID {id_}'. "
                    "A confirmação vale só para este item e não significa que ele seja seguro.")
        e.pedido.append(id_)
        e.confirmacao_pendente = None
        texto = f"Adicionado: {r['item']} (ID {id_}). Itens no pedido: {len(e.pedido)}."
        if avisos:
            texto += "\n" + "\n".join(dict.fromkeys(avisos))
        if e.alergias or e.intolerancias:
            texto += "\n" + AVISO_ALERGIA
        return texto

    def _remover(self, id_):
        e = self.estado
        if id_ not in e.pedido:
            return f"{self._item(id_)['item']} (ID {id_}) não está no seu pedido."
        e.pedido.remove(id_)
        return f"Removido: {self._item(id_)['item']} (ID {id_}). Itens no pedido: {len(e.pedido)}."

    def _responder_nutriente(self, a):
        campos = a.campos_pergunta
        ids = []
        if a.id_ref is not None or a.ordinal is not None or a.nome_ref is not None or a.erro_referencia or a.nomes_ambiguos:
            id_, erro = self._resolver_referencia(a)
            if erro:
                return erro
            ids = [id_]
        elif self.estado.ultimas_recomendacoes:
            ids = list(self.estado.ultimas_recomendacoes)
        else:
            return "De qual produto? Informe o ID, o nome exato ou peça recomendações antes."
        linhas = []
        for i in ids:
            r = self._item(i)
            partes = []
            for c in campos:
                if c == "porcao_texto":
                    partes.append("porção " + (r["porcao_texto"] or "não informada"))
                elif pd.isna(r[c]):
                    partes.append(f"{CAMPOS_NUTRI[c][1]} não informado")
                else:
                    partes.append(_rotulo_valor(c, r[c]))
            linhas.append(f"{r['item']} (ID {i}): " + "; ".join(partes))
        return "Segundo o menu.csv:\n" + "\n".join(linhas)

    def _responder_composicao(self, a):
        e = self.estado
        tem_ref = a.id_ref is not None or a.ordinal is not None or a.nome_ref is not None or a.erro_referencia or a.nomes_ambiguos
        if not tem_ref:
            return AVISO_ALERGIA if (e.alergias or e.intolerancias or re.search(r"alerg|gluten|lactose", " ".join(a.termos_busca))) else \
                "O CSV não traz ingredientes. Posso informar o que o nome e a categoria indicam se você citar o ID ou o nome do item."
        id_, erro = self._resolver_referencia(a)
        if erro:
            return erro
        r = self._item(id_)
        rotulo = {"indicado": "indicado pelo nome", "possivel": "não pode ser descartado (categoria/tipo)",
                  "ausencia_indicada": "o nome indica ausência", "nao_indicado": "não indicado no nome (não verificado)"}
        partes = [f"{NOME_PROTEINA[p]}: {rotulo[r[f'proteina_{p}']]}" for p in PROTEINAS]
        alerg = ", ".join(r["alergenos_nome"]) or "nenhum indicado no nome"
        bloqueios, _ = avaliar_item(self.base, e, id_)
        texto = (f"{r['item']} (ID {id_}) — o CSV não lista ingredientes. Pelo nome/categoria: " + "; ".join(partes) +
                 f". Alergênicos indicados pelo nome: {alerg}.")
        if bloqueios:
            texto += " Com suas restrições atuais, este item está bloqueado: " + "; ".join(bloqueios) + "."
        return texto + "\n" + AVISO_ALERGIA

    # ---------- recomendação ----------
    def _recomendar(self, termos, notas):
        e = self.estado
        tabela = self.base.df
        candidatos = tabela
        if e.categoria_atual:
            candidatos = candidatos[candidatos["grupo"] == e.categoria_atual]
        if e.subtipo_atual:
            if e.categoria_atual == "lanche":
                candidatos = candidatos[candidatos["subtipo"] == e.subtipo_atual]
            elif e.subtipo_atual in PROTEINAS:
                candidatos = candidatos[candidatos[f"proteina_{e.subtipo_atual}"] == "indicado"]
            elif e.subtipo_atual == "nuggets":
                candidatos = candidatos[candidatos["nome_norm"].str.contains("nugget")]
        avaliados = {int(i): avaliar_item(self.base, e, i) for i in candidatos["id_menu"]}
        permitidos = [i for i, (b, _) in avaliados.items() if not b]
        prefixo = ("\n".join(notas) + "\n") if notas else ""
        rotulo = NOME_GRUPO.get(e.categoria_atual)
        onde = (f"de {rotulo}" if rotulo else "do cardápio") + (f" ({e.subtipo_atual})" if e.subtipo_atual else "")
        if not permitidos:
            self._invalidar_lista("a última consulta não teve resultados")
            return (prefixo + f"Não encontrei itens compatíveis {onde} no menu.csv "
                    f"({len(avaliados)} avaliados; restrições: {_descrever_restricoes(e)}). "
                    "Posso mudar a categoria ou flexibilizar algum limite.")
        consulta = consulta_para_busca(termos)
        especifica = bool(consulta.strip())
        escores = dict(buscar_similares(self.base.indice, consulta, permitidos, k=len(permitidos))) if especifica else {}
        if especifica and max(escores.values(), default=0.0) <= 0:
            self._invalidar_lista("a última consulta não teve resultados")
            return (prefixo + f"Não encontrei '{' '.join(termos)}' entre os itens compatíveis {onde} no menu.csv. "
                    "Tente outro nome (os nomes do CSV estão em inglês) ou peça uma categoria.")
        selecao = tabela.loc[permitidos].copy()
        selecao["similaridade"] = selecao["id_menu"].map(escores).fillna(0.0)
        if especifica:
            selecao = selecao[selecao["similaridade"] > 0]
        campos, crescente = [], []
        for pref, (_, coluna) in PREFERENCIAS_REL.items():
            if e.preferencias.get(pref):
                campos.append(coluna); crescente.append(pref != "prefere_mais_proteina")
        campos += ["similaridade", "id_menu"]; crescente += [False, True]
        itens = selecao.sort_values(campos, ascending=crescente, na_position="last").head(5)
        e.ultimas_recomendacoes = [int(i) for i in itens["id_menu"]]
        e.motivo_lista_invalida = ""
        e.confirmacao_pendente = None
        avisos = []
        for i in e.ultimas_recomendacoes:
            avisos += avaliados[i][1]
        linhas = [f"{n}. " + formatar_item(r) for n, r in enumerate(itens.to_dict("records"), start=1)]
        texto = prefixo + f"Encontrei estas opções {onde} no menu.csv:\n" + "\n".join(linhas)
        if e.limites or e.exclusoes or e.dieta:
            texto += f"\nRestrições aplicadas: {_descrever_restricoes(e)}."
        if avisos:
            texto += "\n" + "\n".join(dict.fromkeys(avisos))
        if e.alergias or e.intolerancias:
            texto += "\n" + AVISO_ALERGIA
        return texto + "\nQuer adicionar algum ao pedido (ex.: 'adicione o segundo' ou 'adicione ID N'), ver outra categoria ou ajustar suas preferências?"

    # ---------- turno ----------
    def responder(self, mensagem):
        verificar_base(self.base)
        texto = str(mensagem or "").strip()
        if not texto:
            return "Pode escrever o que você procura? Ex.: 'quero um lanche de frango'."
        e = self.estado
        a = analisar_mensagem(texto, self.base)
        e.historico.append({"usuario": texto})
        if a.reiniciar:
            self.estado = EstadoPedido()
            return "Pedido reiniciado. Você prefere lanche, sobremesa, bebida, acompanhamento ou café da manhã?"

        comando_local = (a.ver_pedido or a.comando is not None or (a.confirmacao and e.confirmacao_pendente is not None)
                         or bool(a.campos_pergunta) or a.pergunta_composicao)
        llm, falha_llm = None, None
        if not comando_local and self.interpretar_fn is not None:
            try:
                llm = self.interpretar_fn(texto)
                if not isinstance(llm, Interpretacao):
                    llm = Interpretacao.model_validate(llm)
            except Exception as erro:
                falha_llm = type(erro).__name__
                llm = None
        mudou, notas = self._aplicar_restricoes(a, llm)
        if falha_llm:
            notas.insert(0, f"(O modelo de linguagem não respondeu: {falha_llm}. Usei só a interpretação local; nada foi alterado no pedido.)")
        for problema in a.limite_invalido:
            notas.append(f"Limite ignorado: {problema}.")
        for op, valor in a.limite_ambiguo:
            if (op, valor) not in e.limites.values():
                notas.append(f"Não apliquei o limite '{TEXTO_OP[op]} {valor:g}': ele se refere a calorias, sódio (mg) "
                             f"ou açúcar (g)? Ex.: '{TEXTO_OP[op]} {valor:g} calorias'.")
        if mudou and e.pedido:
            conflitos = self._conflitos_pedido()
            if conflitos:
                notas.append(conflitos)
        if a.informou_restricao or (llm is not None and llm.restricoes_informadas):
            e.restricoes_respondidas = True

        # 1) Comandos locais
        if a.ver_pedido:
            return "\n".join(notas + [self._ver_pedido()])
        if a.confirmacao and e.confirmacao_pendente is not None:
            pendente = e.confirmacao_pendente
            if a.id_ref is not None and a.id_ref != pendente:
                return f"A confirmação pendente é para o ID {pendente}. Escreva 'confirmo adicionar ID {pendente}'."
            e.confirmacao_pendente = pendente
            return "\n".join(notas + [self._adicionar(pendente)])
        if a.confirmacao and not a.comando and e.confirmacao_pendente is None and (e.alergias or e.intolerancias):
            notas.append("Registrei sua ciência sobre a limitação do CSV. Isso não torna nenhum item seguro e não autoriza adições automaticamente.")
        intencao_llm = llm.intencao if llm is not None else None
        comando = a.comando or (intencao_llm if intencao_llm in ("adicionar", "remover") else None)
        if comando:
            if a.comando is None and llm is not None and llm.item_referido:
                a2 = analisar_mensagem(llm.item_referido, self.base)
                for campo in ("id_ref", "nome_ref", "nomes_ambiguos", "erro_referencia"):
                    if getattr(a, campo) in (None, []) and getattr(a2, campo) not in (None, []):
                        setattr(a, campo, getattr(a2, campo))
            id_, erro = self._resolver_referencia(a)
            if erro:
                return "\n".join(notas + [erro])
            return "\n".join(notas + [self._adicionar(id_) if comando == "adicionar" else self._remover(id_)])
        if intencao_llm == "ver_pedido":
            return "\n".join(notas + [self._ver_pedido()])
        if intencao_llm == "reiniciar":
            return "\n".join(notas + ["Para apagar o pedido e as preferências, escreva 'reiniciar pedido'."])
        if a.campos_pergunta:
            return "\n".join(notas + [self._responder_nutriente(a)])
        if a.pergunta_composicao:
            return "\n".join(notas + [self._responder_composicao(a)])

        # 2) Navegação e recomendação
        notas += self._atualizar_navegacao(a, llm)
        termos = a.termos_busca
        if not e.restricoes_respondidas:
            if e.perguntas_restricao == 0:
                e.perguntas_restricao = 1
                e.termos_pendentes = termos
                return "\n".join(notas + ["Antes de recomendar, você tem alguma preferência, restrição alimentar "
                                          "(ex.: não come carne bovina), limite nutricional ou alergia? "
                                          "Pode dizer \"sem restrições e sem alergias\"."])
            e.restricoes_respondidas = True
            notas.append("Sigo sem restrições informadas. Você pode informá-las a qualquer momento.")
        if not termos and e.termos_pendentes:
            termos = e.termos_pendentes
        e.termos_pendentes = []
        if not e.categoria_atual and not termos:
            if mudou or a.informou_restricao:
                notas.append("Anotado. Restrições atuais: " + _descrever_restricoes(e) + ".")
            return "\n".join(notas + ["Você prefere lanche (frango, bovino, porco, peixe ou nuggets), sobremesa, bebida, acompanhamento ou café da manhã?"])
        if e.categoria_atual == "lanche" and not e.subtipo_atual and re.search(r"quais tipos|divisoes|tipos de lanche", normalizar(texto)):
            return "Divisões de lanches: frango, bovino, porco, peixe, nuggets e outros. Qual você prefere?"
        return self._recomendar(termos, notas)

assistente = AssistenteMcDonalds(base, interpretar_fn=interpretar_llm if interpretador is not None else None)
print("Assistente inicializado", "(com LLM)." if interpretador is not None else "(sem LLM: apenas interpretação local).")

**Demonstração sem custo (sem LLM).** A célula abaixo roda a conversa da auditoria apenas com a interpretação local. Ela não chama o modelo e serve para conferir, no Colab, a regra de exclusão bovina, a adição bloqueada, o limite numérico e o aviso de alergia.

In [ ]:
demo = AssistenteMcDonalds(base, interpretar_fn=None)
for mensagem in ["quero lanche", "eu nao como carne de vaca", "adicione ID 42",
                 "quero sobremesa com menos de 300 calorias", "adicione o segundo",
                 "tenho alergia a leite", "adicione ID 101", "ver meu pedido"]:
    print("Você:", mensagem)
    print("Assistente:", demo.responder(mensagem), "\n")

Interface de conversa no Colab

Execute a célula abaixo para **definir** a função e, em seguida, rode `conversar()` (descomente a última linha ou execute em uma nova célula). Digite sair para encerrar. Exemplos: “quero um lanche”, “frango com menos sódio”, “adicione ID 66”, “adicione o segundo”, “quantas calorias tem o Big Mac?”, “mostre meu pedido”.

In [ ]:
def conversar(assistente_conversa=None):
    assistente_conversa = assistente_conversa or assistente
    print("Assistente McDonald's (baseado no menu.csv). Digite 'sair' para encerrar.")
    while True:
        entrada = input("Você: ").strip()
        if normalizar(entrada) in {"sair", "encerrar", "exit"}:
            break
        if not entrada:
            continue
        try:
            print("Assistente:", assistente_conversa.responder(entrada), "\n")
        except Exception as erro:
            print("Falha ao responder:", type(erro).__name__, str(erro)[:300])

# A conversa não inicia automaticamente ao "Executar tudo". Descomente para iniciar:
# conversar()